# MLOps：視聴データからF1層の有無を予測する

5局の合成視聴データを観察し、XGBoostを学習・評価します。Model Registryへ登録した同じモデルで、判断理由の説明、20,000台の予測、結果保存までを行います。

モデル本体、版、評価指標、実行環境、予測関数、説明関数をSnowflake上でまとめて管理できることを確認します。Markdown説明の下にあるPythonコードセルを上から1つずつ実行してください。

## 1. Notebookを準備する

**このセルで行うこと**

- Snowflakeのセッションを取得する
- 使用するロール・ウェアハウス・データベースを設定する
- pandas、scikit-learn、XGBoost、Snowpark、Snowflake MLを読み込む
- 登録するモデル名 `TV_F1_PRESENCE_MODEL` と初回の版 `V1` を設定する

この章で使うモデルは **XGBoost** です。複数の決定木を組み合わせ、F1層がいる確率を0〜1で計算する分類モデルです。

**実行**

次のコードセルを実行します。

**成功の目印**

`準備完了: TV_F1_PRESENCE_MODEL V1` が表示されます。

In [ ]:
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import plotly.express as px
import sklearn
import xgboost
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from snowflake.ml.registry import Registry
from snowflake.snowpark import functions as sf
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import DoubleType, StringType, StructField, StructType
from xgboost import XGBClassifier

session = get_active_session()
for variable_name in ('model', 'registered_version', 'prediction_pdf'):
    globals().pop(variable_name, None)
if session.get_current_role().strip('"') != 'BCAST_PLATFORM_ENGINEER_ROLE':
    raise RuntimeError('NotebookのロールをBCAST_PLATFORM_ENGINEER_ROLEに変更してください。')
session.use_warehouse('BCAST_PLATFORM_COMMON_WH')
session.use_database('BCAST_PLATFORM_HANDSON')
session.use_schema('ML')

MODEL_NAME = 'TV_F1_PRESENCE_MODEL'
MODEL_VERSION = 'V1'
DATASET_VERSION = 'F1_SIGNAL_V2'
THRESHOLD = 0.50
SPLIT_SEED = 20260918
GENRES = ['NEWS', 'DRAMA', 'VARIETY', 'ANIME', 'SPORTS', 'MUSIC', 'MOVIE', 'INFO']
FEATURE_COLUMNS = [genre + '_SHARE' for genre in GENRES] + [
    'TOTAL_MINUTES',
    'TOTAL_SESSIONS',
    'ACTIVE_DAYS',
    'MEAN_MINUTES',
]

def collect_features(daily_source):
    features = daily_source.group_by('DEVICE_ID').agg(
        sf.sum('VIEW_MINUTES').cast('double').alias('TOTAL_MINUTES'),
        sf.sum('SESSION_COUNT').cast('double').alias('TOTAL_SESSIONS'),
        sf.count_distinct('VIEW_DATE').cast('double').alias('ACTIVE_DAYS'),
        *[
            sf.sum(
                sf.when(
                    sf.col('GENRE') == genre,
                    sf.col('VIEW_MINUTES'),
                ).otherwise(sf.lit(0))
            ).cast('double').alias(genre + '_MINUTES')
            for genre in GENRES
        ],
    )
    for genre in GENRES:
        features = features.with_column(
            genre + '_SHARE',
            sf.col(genre + '_MINUTES') / sf.nullif(sf.col('TOTAL_MINUTES'), sf.lit(0)),
        )
    features = features.with_column(
        'MEAN_MINUTES',
        sf.col('TOTAL_MINUTES') / sf.nullif(sf.col('TOTAL_SESSIONS'), sf.lit(0)),
    )
    return features.select('DEVICE_ID', *FEATURE_COLUMNS).sort('DEVICE_ID').to_pandas()

print('準備完了:', MODEL_NAME, MODEL_VERSION)

## 2. 予測に使うデータを準備する

**入力**

第2章で作成した5局共通の日次視聴データを使います。

**このセルで行うこと**

テレビ1台につき1行へ集計し、モデルへ渡す12項目を作ります。

- 8ジャンルの視聴割合
- 総視聴時間
- 視聴回数
- 視聴した日数
- 1回あたりの平均視聴時間

20,000台のうち、正解ラベルがある2,000台を学習と評価に使います。

**実行**

次のコードセルを実行します。

**成功の目印**

`データ準備完了: 全体 20000台 / 学習・評価対象 2000台` と、先頭5行が表示されます。

In [ ]:
globals().pop('registered_version', None)
globals().pop('prediction_pdf', None)

features_pdf = collect_features(
    session.table('BCAST_PLATFORM_HANDSON.COMMON.VIEWING_DAILY')
)
features_pdf[FEATURE_COLUMNS] = features_pdf[FEATURE_COLUMNS].astype('float64')

if (
    len(features_pdf) != 20000
    or features_pdf['DEVICE_ID'].isna().any()
    or not features_pdf['DEVICE_ID'].is_unique
):
    raise ValueError('特徴量は20,000台につき1行必要です。第2章のdbt buildを確認してください。')
if not np.isfinite(features_pdf[FEATURE_COLUMNS].to_numpy()).all():
    raise ValueError('特徴量に欠損または無限大があります。')

labels_pdf = (
    session.table('BCAST_PLATFORM_HANDSON.RAW.DEVICE_LABELS')
    .filter(sf.col('LABEL_AVAILABLE'))
    .select('DEVICE_ID', 'TARGET_F1')
    .sort('DEVICE_ID')
    .to_pandas()
)
if len(labels_pdf) != 2000 or not labels_pdf['DEVICE_ID'].is_unique:
    raise ValueError('学習・評価用の正解データは2,000台必要です。')
if labels_pdf['TARGET_F1'].isna().any() or not labels_pdf['TARGET_F1'].isin([0, 1]).all():
    raise ValueError('TARGET_F1には0または1が必要です。')

known_pdf = (
    features_pdf
    .merge(labels_pdf, on='DEVICE_ID', validate='one_to_one')
    .sort_values('DEVICE_ID')
    .reset_index(drop=True)
)
if len(known_pdf) != 2000:
    raise ValueError('特徴量と正解データを2,000台分結合できません。')

print('データ準備完了: 全体', len(features_pdf), '台 / 学習・評価対象', len(known_pdf), '台')
features_pdf.head()

## 3. 学習データを観察する（EDA）

機械学習を始める前に、学習に使う1,600台のデータを観察します。これをEDA（探索的データ分析）と呼びます。

**このセルで見ること**

- F1層なし・ありの台数に偏りがあるか
- 主な視聴項目の値がどのように分布しているか
- 2つのグループで視聴傾向に違いがあるか

**実行**

次のコードセルを実行します。

**成功の目印**

`学習データのグループ構成` と `主な視聴項目の分布` の2つのグラフ、およびグループ別平均の表が表示されます。

In [ ]:
import plotly.express as px

# EDA用に、正解が分かっている2,000台から学習用1,600台を分ける
train_indices, _ = train_test_split(
    np.arange(len(known_pdf)),
    test_size=0.2,
    random_state=SPLIT_SEED,
    stratify=known_pdf.TARGET_F1,
)
eda_pdf = known_pdf.iloc[train_indices].copy()
eda_pdf["GROUP"] = eda_pdf["TARGET_F1"].map({0: "F1層なし", 1: "F1層あり"})

# 正解ラベルの内訳をグラフで確認
label_counts = (
    eda_pdf["GROUP"]
    .value_counts()
    .rename_axis("GROUP")
    .reset_index(name="TV_COUNT")
)
label_fig = px.bar(
    label_counts,
    x="GROUP",
    y="TV_COUNT",
    color="GROUP",
    title="学習データのグループ構成",
    labels={"GROUP": "正解ラベル", "TV_COUNT": "テレビ台数"},
    color_discrete_map={"F1層なし": "#7F8C8D", "F1層あり": "#29B5E8"},
    template="plotly_white",
)
label_fig.update_layout(showlegend=False)
label_fig.show()

# 予測に使う項目のうち、理解しやすい4項目の分布を確認
eda_distribution_columns = [
    "DRAMA_SHARE",
    "ANIME_SHARE",
    "TOTAL_MINUTES",
    "ACTIVE_DAYS",
]
eda_long_pdf = eda_pdf.melt(
    id_vars="GROUP",
    value_vars=eda_distribution_columns,
    var_name="FEATURE",
    value_name="VALUE",
)
distribution_fig = px.box(
    eda_long_pdf,
    x="GROUP",
    y="VALUE",
    color="GROUP",
    facet_col="FEATURE",
    facet_col_wrap=2,
    points="outliers",
    title="主な視聴項目の分布",
    labels={"GROUP": "正解ラベル", "VALUE": "値", "FEATURE": "視聴項目"},
    color_discrete_map={"F1層なし": "#7F8C8D", "F1層あり": "#29B5E8"},
    template="plotly_white",
)
distribution_fig.for_each_annotation(
    lambda annotation: annotation.update(text=annotation.text.split("=")[-1])
)
distribution_fig.update_yaxes(matches=None)
distribution_fig.update_layout(height=650, showlegend=False)
distribution_fig.show()

# 分かりやすい5項目についてグループ別の平均を比較
eda_columns = ["DRAMA_SHARE", "VARIETY_SHARE", "ANIME_SHARE", "TOTAL_MINUTES", "ACTIVE_DAYS"]
group_means = (
    eda_pdf
    .groupby("GROUP")[eda_columns]
    .mean()
    .round(3)
)
print("グループ別の平均")
display(group_means)

## 4. ジャンル別の視聴傾向をグラフで見る

学習に使う1,600台について、F1層なし・ありの2グループで8ジャンルの平均視聴割合を比べます。

棒の高さに違いがあれば、その視聴傾向をモデルが予測の手がかりとして学習できます。モデルは1項目だけで判断せず、12項目を組み合わせます。

**実行**

次のコードセルを実行します。

**成功の目印**

2グループのジャンル別平均を並べた棒グラフが表示されます。

In [ ]:
# 学習データ1,600台の8ジャンル平均をグラフ用の縦長データへ変換
genre_columns = [genre + "_SHARE" for genre in GENRES]
genre_means = (
    eda_pdf
    .assign(GROUP=eda_pdf["TARGET_F1"].map({0: "F1層なし", 1: "F1層あり"}))
    .groupby("GROUP")[genre_columns]
    .mean()
    .reset_index()
    .melt(id_vars="GROUP", var_name="GENRE", value_name="AVERAGE_SHARE")
)
genre_means["GENRE"] = genre_means["GENRE"].str.replace("_SHARE", "", regex=False)

fig = px.bar(
    genre_means,
    x="GENRE",
    y="AVERAGE_SHARE",
    color="GROUP",
    barmode="group",
    title="学習データのジャンル別平均視聴割合",
    labels={"GENRE": "ジャンル", "AVERAGE_SHARE": "平均視聴割合", "GROUP": "グループ"},
    template="plotly_white",
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

## 5. XGBoostモデルを学習・評価する

ここから機械学習を行います。

**入力**

- 1台につき12項目の視聴傾向
- 正解ラベル（F1層なし=0、あり=1）

**このセルで行うこと**

- 2,000台を学習用1,600台、評価用400台に分ける
- 学習用データから、12項目と正解の関係をXGBoostに学ばせる
- 学習に使わなかった評価用400台で予測性能を確認する

XGBoostは、小さな決定木を順番に作り、前の木が間違えた部分を次の木で補いながら分類します。学習後のモデルは、12項目を受け取るとF1層がいる確率を0〜1で返します。

**実行**

次のコードセルを実行します。

**成功の目印**

- `学習データ: 1600台 / 評価データ: 400台`
- `モデル: XGBoost`
- `評価完了` と評価結果の表
- `評価データの混同行列` という図

混同行列では、縦が実際の正解、横がモデルの予測です。左上と右下は正解、右上と左下は誤判定の台数を表します。

が表示されます。

In [ ]:
globals().pop('registered_version', None)
globals().pop('prediction_pdf', None)

train_indices, test_indices = train_test_split(
    np.arange(len(known_pdf)),
    test_size=0.2,
    random_state=SPLIT_SEED,
    stratify=known_pdf['TARGET_F1'],
)
train_pdf = known_pdf.iloc[train_indices].copy()
test_pdf = known_pdf.iloc[test_indices].copy()
if len(train_pdf) != 1600 or len(test_pdf) != 400:
    raise ValueError('学習データ1,600台、評価データ400台に分割できません。')

train_features = train_pdf[FEATURE_COLUMNS]
train_labels = train_pdf['TARGET_F1'].astype('int64')
test_features = test_pdf[FEATURE_COLUMNS]
test_labels = test_pdf['TARGET_F1'].astype('int64')

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=1,
)
model.fit(train_features, train_labels)

test_probability = model.predict_proba(test_features)[:, 1]
test_prediction = (test_probability >= THRESHOLD).astype('int64')
metrics_df = pd.DataFrame([{
    'ROC_AUC': roc_auc_score(test_labels, test_probability),
    'PRECISION': precision_score(test_labels, test_prediction, zero_division=0),
    'RECALL': recall_score(test_labels, test_prediction, zero_division=0),
    'F1_SCORE': f1_score(test_labels, test_prediction, zero_division=0),
}])

print('学習データ:', len(train_pdf), '台 / 評価データ:', len(test_pdf), '台')
print('モデル: XGBoost')
print('評価完了')
learner_metrics = (
    metrics_df[['ROC_AUC', 'PRECISION', 'RECALL']]
    .rename(columns={
        'ROC_AUC': 'ROC AUC',
        'PRECISION': '適合率',
        'RECALL': '再現率',
    })
    .round(3)
)
display(learner_metrics)

xgboost_confusion = confusion_matrix(test_labels, test_prediction, labels=[0, 1])
confusion_fig = px.imshow(
    xgboost_confusion,
    x=['予測: F1層なし', '予測: F1層あり'],
    y=['実際: F1層なし', '実際: F1層あり'],
    text_auto=True,
    color_continuous_scale='Blues',
    title='評価データの混同行列',
    labels={'x': 'モデルの予測', 'y': '実際の正解', 'color': 'テレビ台数'},
    template='plotly_white',
)
confusion_fig.update_traces(textfont={'size': 20})
confusion_fig.update_layout(height=430, width=560)
confusion_fig.show()

## 6. モデルをSnowflakeへ登録する

**このセルで行うこと**

学習済みのXGBoostモデルを、`TV_F1_PRESENCE_MODEL` の`V1`としてModel Registryへ登録します。`V1`が既にある場合は、このセルで既存版を削除してから同じ`V1`を登録します。そのため、初回も再実行も同じ手順です。学習データの一部も説明用の基準として一緒に登録します。

Model Registryでは、モデル本体だけでなく、版、評価指標、実行環境、予測関数、説明関数をSnowflake上でまとめて管理できます。

**実行**

次のコードセルを実行します。

**成功の目印**

`モデル登録完了: TV_F1_PRESENCE_MODEL V1` と、`predict`、`predict_proba`、`explain` が表示されます。

In [ ]:
globals().pop('prediction_pdf', None)

registry = Registry(
    session=session,
    database_name='BCAST_PLATFORM_HANDSON',
    schema_name='ML',
)

existing_models = registry.show_models()
existing_models.columns = [str(column).upper() for column in existing_models.columns]
if (
    not existing_models.empty
    and existing_models['NAME'].astype('string').str.upper().eq(MODEL_NAME).any()
):
    existing_model = registry.get_model(MODEL_NAME)
    existing_versions = existing_model.show_versions()
    existing_versions.columns = [str(column).upper() for column in existing_versions.columns]
    if existing_versions['NAME'].astype('string').str.upper().eq(MODEL_VERSION).any():
        existing_model.delete_version(MODEL_VERSION)
        print('既存モデル版を削除:', MODEL_NAME, MODEL_VERSION)

registered_metrics = {
    'test_roc_auc': float(metrics_df.at[0, 'ROC_AUC']),
    'test_precision': float(metrics_df.at[0, 'PRECISION']),
    'test_recall': float(metrics_df.at[0, 'RECALL']),
    'test_f1_score': float(metrics_df.at[0, 'F1_SCORE']),
}
registered_version = registry.log_model(
    model,
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    sample_input_data=train_features.head(200),
    conda_dependencies=[
        'xgboost==' + xgboost.__version__,
        'scikit-learn==' + sklearn.__version__,
    ],
    target_platforms=['WAREHOUSE'],
    python_version='3.12',
    options={
        'target_methods': ['predict', 'predict_proba'],
        'relax_version': False,
        'enable_explainability': True,
    },
    metrics=registered_metrics,
    comment=json.dumps({
        'dataset_version': DATASET_VERSION,
        'threshold': THRESHOLD,
        'algorithm': 'XGBoost',
    }),
)
registered_functions = registered_version.show_functions()

function_names = set()
for function in registered_functions:
    if isinstance(function, dict):
        function_name = next(
            (value for key, value in function.items() if str(key).lower() == 'name'),
            None,
        )
    else:
        function_name = getattr(function, 'name', None)
    if function_name:
        function_names.add(str(function_name).lower())

if not {'predict', 'predict_proba', 'explain'}.issubset(function_names):
    raise RuntimeError('predict、predict_proba、explainの登録を確認できません。')

print('モデル登録完了:', MODEL_NAME, MODEL_VERSION)
registered_functions

## 7. 登録モデルの判断理由をSnowflakeで説明する

**このセルで行うこと**

Model Registryへ登録された `explain` 関数を、評価用データ100台に対してSnowflake上で実行します。

- 全体表示: 1点をテレビ1台として、各項目のSHAP値の分布を表示
- 1台の表示: 各項目が「F1層あり」の方向、または「なし」の方向へどれだけ働いたか

全体表示では、横軸がSHAP値、点の色が元の特徴量値です。赤い点ほどその項目の値が高く、青い点ほど低いことを表します。右側の点は「F1層あり」、左側の点は「なし」の方向へ予測を動かしています。

SHAP値は確率の増減そのものではなく、XGBoost内部の生のモデルスコアへの寄与です。因果関係を示すものではありません。

**成功の目印**

`登録モデルのSHAP分布` と `1台の予測を動かした項目` の2つのグラフが表示されます。

In [ ]:
explain_input_pdf = test_pdf[FEATURE_COLUMNS].head(100).reset_index(drop=True)
explain_schema = StructType([StructField(column, DoubleType()) for column in FEATURE_COLUMNS])
explain_input_sdf = session.create_dataframe(
    explain_input_pdf.to_numpy().tolist(),
    schema=explain_schema,
)
explain_output = (
    registered_version
    .run(explain_input_sdf, function_name='explain')
    .limit(101)
    .to_pandas()
)
explain_output.columns = [str(column).upper() for column in explain_output.columns]
explanation_columns = [column for column in explain_output.columns if column.endswith('_EXPLANATION')]
expected_explanation_columns = {column + '_EXPLANATION' for column in FEATURE_COLUMNS}
if (
    len(explain_output) != 100
    or set(explanation_columns) != expected_explanation_columns
    or not set(FEATURE_COLUMNS).issubset(explain_output.columns)
):
    raise RuntimeError('SHAP説明の行数または列が登録モデルと一致しません。')

shap_pdf = explain_output[explanation_columns].astype('float64').rename(
    columns=lambda column: column.removesuffix('_EXPLANATION')
)
explain_feature_pdf = explain_output[FEATURE_COLUMNS].astype('float64')
if (
    not np.isfinite(shap_pdf.to_numpy()).all()
    or not np.isfinite(explain_feature_pdf.to_numpy()).all()
):
    raise ValueError('SHAP値または元の特徴量値に欠損・無限大があります。')

feature_order = shap_pdf.abs().mean().sort_values(ascending=False).index.tolist()
rng = np.random.default_rng(42)
summary_rows = []
for feature_position, feature in enumerate(feature_order):
    feature_values = explain_feature_pdf[feature]
    value_min = float(feature_values.min())
    value_max = float(feature_values.max())
    if value_max == value_min:
        color_values = np.full(len(feature_values), 0.5)
    else:
        color_values = (feature_values - value_min) / (value_max - value_min)

    for sample_number, (shap_value, feature_value, color_value) in enumerate(
        zip(shap_pdf[feature], feature_values, color_values),
        start=1,
    ):
        summary_rows.append({
            'FEATURE': feature,
            'SHAP_VALUE': float(shap_value),
            'FEATURE_VALUE': float(feature_value),
            'COLOR_VALUE': float(color_value),
            'Y_POSITION': feature_position + rng.uniform(-0.22, 0.22),
            'SAMPLE': sample_number,
        })

shap_summary_pdf = pd.DataFrame(summary_rows)
shap_fig = px.scatter(
    shap_summary_pdf,
    x='SHAP_VALUE',
    y='Y_POSITION',
    color='COLOR_VALUE',
    custom_data=['FEATURE', 'FEATURE_VALUE', 'SAMPLE'],
    title='登録モデルのSHAP分布（評価用100台）',
    labels={
        'SHAP_VALUE': '生のモデルスコアへの寄与',
        'Y_POSITION': '視聴項目',
        'COLOR_VALUE': '特徴量値',
    },
    color_continuous_scale=[
        (0.0, '#2166AC'),
        (0.5, '#F7F7F7'),
        (1.0, '#B2182B'),
    ],
    range_color=(0, 1),
    template='plotly_white',
)
shap_fig.update_traces(
    marker={'size': 7, 'opacity': 0.75},
    hovertemplate=(
        '項目=%{customdata[0]}<br>'
        'SHAP値=%{x:.4f}<br>'
        '元の値=%{customdata[1]:.4f}<br>'
        'サンプル=%{customdata[2]}<extra></extra>'
    ),
)
shap_fig.update_yaxes(
    tickmode='array',
    tickvals=list(range(len(feature_order))),
    ticktext=feature_order,
    autorange='reversed',
)
shap_fig.update_coloraxes(
    colorbar={
        'title': '特徴量値',
        'tickvals': [0, 1],
        'ticktext': ['低い', '高い'],
    }
)
shap_fig.add_vline(x=0, line_color='#7F8C8D', line_dash='dash')
shap_fig.update_layout(height=620)
shap_fig.show()

example_position = int(np.argmax(model.predict_proba(explain_input_pdf)[:, 1]))
example_device_id = test_pdf.iloc[example_position]['DEVICE_ID']
example_input_pdf = explain_input_pdf.iloc[[example_position]].reset_index(drop=True)
example_input_sdf = session.create_dataframe(
    example_input_pdf.to_numpy().tolist(),
    schema=explain_schema,
)
example_output = (
    registered_version
    .run(example_input_sdf, function_name='explain')
    .limit(2)
    .to_pandas()
)
example_output.columns = [str(column).upper() for column in example_output.columns]
if len(example_output) != 1 or set(column for column in example_output.columns if column.endswith('_EXPLANATION')) != expected_explanation_columns:
    raise RuntimeError('1台分のSHAP説明が登録モデルと一致しません。')
example_shap_pdf = (
    example_output[explanation_columns]
    .astype('float64')
    .rename(columns=lambda column: column.removesuffix('_EXPLANATION'))
    .iloc[0]
    .rename('SHAP_VALUE')
    .rename_axis('FEATURE')
    .reset_index()
    .sort_values('SHAP_VALUE')
)
example_shap_pdf['DIRECTION'] = np.where(
    example_shap_pdf['SHAP_VALUE'] >= 0,
    'F1層ありの方向',
    'F1層なしの方向',
)
example_fig = px.bar(
    example_shap_pdf,
    x='SHAP_VALUE',
    y='FEATURE',
    color='DIRECTION',
    orientation='h',
    title=f'1台の予測を動かした項目: {example_device_id}',
    labels={
        'SHAP_VALUE': '生のモデルスコアへの寄与',
        'FEATURE': '視聴項目',
        'DIRECTION': '方向',
    },
    color_discrete_map={
        'F1層ありの方向': '#29B5E8',
        'F1層なしの方向': '#7F8C8D',
    },
    template='plotly_white',
)
example_fig.show()
print('Snowflake上の登録モデルでSHAP説明完了:', len(shap_pdf), '台')

## 8. 登録したモデルで20,000台を予測する

**入力**

20,000台それぞれの12項目の視聴傾向を、Model Registryへ登録したモデルに渡します。

**出力**

- `PROB_F1`: F1層がいる確率（0〜1）
- `PREDICTED_HAS_F1`: 確率が0.50以上なら1、未満なら0

**実行**

次のコードセルを実行します。

**成功の目印**

`予測完了: 20000台`、判定別の台数、および判定0・1をそれぞれ5行ずつ含む予測例が表示されます。

In [ ]:
prediction_input = features_pdf[['DEVICE_ID'] + FEATURE_COLUMNS].reset_index(drop=True)
prediction_schema = StructType(
    [StructField('DEVICE_ID', StringType())]
    + [StructField(column, DoubleType()) for column in FEATURE_COLUMNS]
)
prediction_sdf = session.create_dataframe(
    prediction_input.to_numpy().tolist(),
    schema=prediction_schema,
)
registry_output = (
    registered_version
    .run(prediction_sdf, function_name='predict_proba')
    .to_pandas()
)
registry_output.columns = [str(column).upper() for column in registry_output.columns]

required_output_columns = {'DEVICE_ID', 'OUTPUT_FEATURE_1'}
if len(registry_output) != 20000 or not required_output_columns.issubset(registry_output.columns):
    raise RuntimeError('登録モデルから20,000台分の確率を取得できません。')

prediction_pdf = registry_output[
    ['DEVICE_ID', 'OUTPUT_FEATURE_1']
].rename(columns={'OUTPUT_FEATURE_1': 'PROB_F1'})
prediction_pdf['PROB_F1'] = prediction_pdf['PROB_F1'].astype('float64')
if not prediction_pdf['DEVICE_ID'].is_unique or prediction_pdf['PROB_F1'].isna().any():
    raise RuntimeError('20,000台すべてに予測確率を対応できません。')
prediction_pdf['PREDICTED_HAS_F1'] = (
    prediction_pdf['PROB_F1'] >= THRESHOLD
).astype('int64')

print('予測完了:', len(prediction_pdf), '台')
prediction_counts = (
    prediction_pdf['PREDICTED_HAS_F1']
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis('PREDICTED_HAS_F1')
    .reset_index(name='TV_COUNT')
)
display(prediction_counts)

prediction_examples = (
    prediction_pdf
    .sort_values(
        ['PREDICTED_HAS_F1', 'PROB_F1'],
        ascending=[True, False],
    )
    .groupby('PREDICTED_HAS_F1', group_keys=False)
    .head(5)
    [['DEVICE_ID', 'PROB_F1', 'PREDICTED_HAS_F1']]
    .reset_index(drop=True)
)
if set(prediction_examples['PREDICTED_HAS_F1']) != {0, 1}:
    raise RuntimeError('判定0・1の両方を予測例として表示できません。')
display(prediction_examples)

## 9. 20,000台の予測結果をテーブルへ保存する

**このセルで行うこと**

直前のセルで作った20,000台分の予測結果を、`BCAST_PLATFORM_HANDSON.ML.PREDICTIONS` テーブルとして保存します。

保存するのは次の情報です。

- 端末ID
- F1層がいる確率
- 確率0.50を基準にした判定
- 予測に使ったモデル名とモデル版
- 予測日時、判定基準、データ版

次のコードセルを1回実行してください。20,000台分の予測結果をテーブルへ作成または置き換えます。

**最終結果**

`BCAST_PLATFORM_HANDSON.ML.PREDICTIONS` に20,000行が保存されます。このテーブルが第4章のStreamlitで使う予測データです。

**成功の目印**

`保存完了: BCAST_PLATFORM_HANDSON.ML.PREDICTIONS 20000行` と、判定0・1をそれぞれ5行ずつ含む保存結果の例が表示されます。

In [ ]:
if 'prediction_pdf' not in globals():
    raise ValueError('先にNotebookセル17で予測を実行してください。')
if len(prediction_pdf) != 20000:
    raise ValueError('保存する予測結果は20,000行必要です。')

output_pdf = prediction_pdf.copy()
output_pdf['MODEL_NAME'] = MODEL_NAME
output_pdf['MODEL_VERSION'] = MODEL_VERSION
output_pdf['PREDICTED_AT'] = datetime.now(timezone.utc).replace(tzinfo=None)
output_pdf['PREDICTION_THRESHOLD'] = float(THRESHOLD)
output_pdf['DATASET_VERSION'] = DATASET_VERSION

session.write_pandas(
    output_pdf,
    table_name='PREDICTIONS',
    database='BCAST_PLATFORM_HANDSON',
    schema='ML',
    auto_create_table=True,
    overwrite=True,
    use_logical_type=True,
)

published_table = 'BCAST_PLATFORM_HANDSON.ML.PREDICTIONS'
saved_preview = session.table(published_table)
saved_count = saved_preview.count()
if saved_count != 20000:
    raise RuntimeError('保存後の行数が20,000行ではありません。')

print('保存完了:', published_table, saved_count, '行')
saved_examples = pd.concat(
    [
        saved_preview
        .filter(sf.col('PREDICTED_HAS_F1') == predicted_class)
        .select(
            'DEVICE_ID',
            'PROB_F1',
            'PREDICTED_HAS_F1',
            'MODEL_VERSION',
        )
        .sort(sf.col('PROB_F1').desc())
        .limit(5)
        .to_pandas()
        for predicted_class in (0, 1)
    ],
    ignore_index=True,
)
if set(saved_examples['PREDICTED_HAS_F1']) != {0, 1}:
    raise RuntimeError('保存結果から判定0・1の両方を表示できません。')
saved_examples

## 10. Notebookサービスを停止する

データの観察、モデルの学習・評価、Model Registryへの登録、20,000台の予測、結果保存まで完了しました。

最後に **Connected → サービス名 → Suspend** を選び、状態が **SUSPENDED** になったことを確認します。